# 07 — Trustworthiness audit

This notebook adds the trustworthiness layert:
1. **validation-only threshold calibration**
2. **leakage-aware ablation**
3. **LIME stability across random seeds**
4. **subgroup robustness audit**


# 0. Config and shared helpers


In [ ]:
from __future__ import annotations

import json
import math
import sys
import warnings
import zipfile
from itertools import combinations
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.options.display.float_format = lambda x: f"{x:.4f}"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd()
    start = start.resolve()

    candidates = [start] + list(start.parents)
    for cand in candidates:
        if (cand / "regression_model").exists() and (cand / "notebooks").exists():
            return cand
        if (cand / "README.md").exists() and (cand / "src").exists():
            return cand
    return start

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

# Prefer the baseline artifact because the artifact shipped inside output.zip
# can be version-sensitive across pandas / xgboost installations.
BASELINE_ARTIFACT_PATH = PROJECT_ROOT / "regression_model" / "models" / "xgb_energy_artifact.joblib"
OUTPUT_ZIP_PATH = PROJECT_ROOT / "notebooks/baseline_implementation" / "output.zip"

BLOG_OUTPUT_DIR = PROJECT_ROOT / "outputs_blogpost"
FIG_DIR = BLOG_OUTPUT_DIR / "figures"
TABLE_DIR = BLOG_OUTPUT_DIR / "tables"

for p in [BLOG_OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RAW_NUMERIC_FEATURES = [
    "duration_min",
    "distance_km",
    "speed_mean",
    "speed_var",
    "accel_mean",
    "accel_var",
    "accel_p95",
    "stop_go_ratio",
    "idle_time_min",
    "fuel_energy_kWh",
    "battery_energy_kWh",
    "ac_energy_kWh",
    "heater_energy_kWh",
    "hv_current_abs_mean",
    "hv_current_abs_p95",
    "hv_voltage_mean",
    "maf_mean",
    "maf_p95",
    "Generalized_Weight",
]

DIRECT_TARGET_COMPONENTS = [
    "fuel_energy_kWh",
    "battery_energy_kWh",
    "ac_energy_kWh",
    "heater_energy_kWh",
]

CATEGORICAL_FEATURES = [
    "VehicleType",
    "Vehicle Class",
    "Transmission",
    "Drive Wheels",
]

TARGET = "energy_per_km"
YHAT_COL = "predicted_energy_per_km"
RESIDUAL_COL = "residual"

def load_scored_table_from_repo() -> pd.DataFrame:
    if not OUTPUT_ZIP_PATH.exists():
        raise FileNotFoundError(
            f"Could not find {OUTPUT_ZIP_PATH}. Place these notebooks into the repository root."
        )

    with zipfile.ZipFile(OUTPUT_ZIP_PATH) as zf:
        inner = "outputs_final/cache/trip_table_scored.csv.gz"
        if inner not in zf.namelist():
            raise FileNotFoundError(f"{inner} not found inside {OUTPUT_ZIP_PATH}")
        raw_gz = zf.read(inner)

    import gzip
    csv_bytes = gzip.decompress(raw_gz)
    from io import BytesIO
    df = pd.read_csv(BytesIO(csv_bytes))
    return df

def load_baseline_artifact() -> dict:
    if not BASELINE_ARTIFACT_PATH.exists():
        raise FileNotFoundError(f"Could not find baseline artifact at {BASELINE_ARTIFACT_PATH}")
    return joblib.load(BASELINE_ARTIFACT_PATH)

def safe_numeric_fill_value(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce")
    median = s.median()
    if not np.isfinite(median):
        return 0.0
    return float(median)

def fill_raw_features(
        df: pd.DataFrame,
        model_numeric_features: list[str],
        categorical_features: list[str],
        numeric_fill: dict[str, float] | None = None,
    ) -> pd.DataFrame:

    selected_columns = list(model_numeric_features) + list(categorical_features)
    out = df.reindex(columns=selected_columns).copy()

    for col in model_numeric_features:
        if col not in out.columns:
            out[col] = np.nan
    for col in categorical_features:
        if col not in out.columns:
            out[col] = "NO DATA"

    if numeric_fill is None:
        numeric_fill = {col: safe_numeric_fill_value(out[col]) for col in model_numeric_features}

    for col in model_numeric_features:
        fill_value = numeric_fill.get(col, 0.0)
        if not np.isfinite(fill_value):
            fill_value = 0.0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(fill_value)

    for col in categorical_features:
        out[col] = out[col].astype("string").fillna("NO DATA")

    return out

def make_design(
        df: pd.DataFrame,
        model_numeric_features: list[str],
        categorical_features: list[str],
        design_columns: list[str] | None = None,
        numeric_fill: dict[str, float] | None = None,
    ) -> tuple[pd.DataFrame, dict[str, float], list[str]]:

    raw = fill_raw_features(
        df,
        model_numeric_features=model_numeric_features,
        categorical_features=categorical_features,
        numeric_fill=numeric_fill,
    )
    
    X = pd.get_dummies(raw, columns=categorical_features, dummy_na=False)

    if design_columns is None:
        design_columns = list(X.columns)
    X = X.reindex(columns=design_columns, fill_value=0.0)

    if numeric_fill is None:
        numeric_fill = {col: safe_numeric_fill_value(raw[col]) for col in model_numeric_features}

    return X.astype(float), numeric_fill, list(design_columns)

def evaluate_regression(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }

def transform_target(y: np.ndarray, transform: str | None) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    if transform is None:
        return y
    if transform == "log1p":
        if np.any(y < 0):
            raise ValueError("log1p requires non-negative targets")
        return np.log1p(y)
    raise ValueError(f"Unsupported transform: {transform}")

def inverse_transform_target(y: np.ndarray, transform: str | None) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    if transform is None:
        return y
    if transform == "log1p":
        out = np.expm1(y)
        return np.clip(out, 0.0, None)
    raise ValueError(f"Unsupported transform: {transform}")

from src.xai.lime import explain_instance

AUDIT_DIR = BLOG_OUTPUT_DIR / "audit"
AUDIT_FIG_DIR = AUDIT_DIR / "figures"
AUDIT_TABLE_DIR = AUDIT_DIR / "tables"

for p in [AUDIT_DIR, AUDIT_FIG_DIR, AUDIT_TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def positive_residual_threshold(residuals: pd.Series, q: float = 0.95) -> float:
    positive = residuals[residuals > 0]
    if positive.empty:
        return 0.0
    return float(positive.quantile(q))

def anomaly_flag(residuals: pd.Series, threshold: float) -> pd.Series:
    return (residuals > threshold).astype(int)

def subgroup_metrics(df: pd.DataFrame, group_col: str, anomaly_col: str) -> pd.DataFrame:
    rows = []
    for value, part in df.groupby(group_col, dropna=False):
        metrics = evaluate_regression(
            part[TARGET].to_numpy(dtype=float),
            part[YHAT_COL].to_numpy(dtype=float),
        )
        rows.append({
            "group_col": group_col,
            "group_value": value,
            "n": int(len(part)),
            "mae": metrics["mae"],
            "rmse": metrics["rmse"],
            "r2": metrics["r2"],
            "anomaly_rate": float(part[anomaly_col].mean()),
            "mean_residual": float(part[RESIDUAL_COL].mean()),
        })

    return pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)

def mean_pairwise_jaccard(feature_lists: list[list[str]]) -> float:
    if len(feature_lists) < 2:
        return 1.0
    
    vals = []

    for a, b in combinations(feature_lists, 2):
        sa, sb = set(a), set(b)
        union = sa | sb
        vals.append(len(sa & sb) / len(union) if union else 1.0)

    return float(np.mean(vals))

def mean_sign_consistency(contribution_frames: list[pd.DataFrame]) -> float:
    if not contribution_frames:
        return np.nan
    
    all_features = sorted(set().union(*[set(df["feature"]) for df in contribution_frames]))
    scores = []

    for feature in all_features:
        signs = []
        for df in contribution_frames:
            part = df.loc[df["feature"] == feature, "contribution"]
            if not part.empty:
                val = float(part.iloc[0])

                if val > 0:
                    signs.append(1)

                elif val < 0:
                    signs.append(-1)

        if not signs:
            continue

        pos = sum(1 for s in signs if s > 0)
        neg = sum(1 for s in signs if s < 0)
        scores.append(max(pos, neg) / len(signs))
        
    return float(np.mean(scores)) if scores else np.nan


## 1. Load the scored table and the baseline artifact


In [ ]:
scored_df = load_scored_table_from_repo()
artifact = load_baseline_artifact()

display(scored_df.head(2))


## 2. Validation-only threshold calibration

In [ ]:
val_df = scored_df.loc[scored_df["split"] == "val"].copy()
test_df = scored_df.loc[scored_df["split"] == "test"].copy()

thr_val95 = positive_residual_threshold(val_df[RESIDUAL_COL], q=0.95)
thr_val98 = positive_residual_threshold(val_df[RESIDUAL_COL], q=0.98)

test_df["is_anomaly_val95"] = anomaly_flag(test_df[RESIDUAL_COL], thr_val95)
test_df["is_anomaly_val98"] = anomaly_flag(test_df[RESIDUAL_COL], thr_val98)

calibration_summary = pd.DataFrame([
    {
        "threshold_name": "val_positive_q95",
        "threshold": thr_val95,
        "test_anomaly_rate": float(test_df["is_anomaly_val95"].mean()),
        "n_test_anomalies": int(test_df["is_anomaly_val95"].sum()),
    },
    {
        "threshold_name": "val_positive_q98",
        "threshold": thr_val98,
        "test_anomaly_rate": float(test_df["is_anomaly_val98"].mean()),
        "n_test_anomalies": int(test_df["is_anomaly_val98"].sum()),
    },
])

display(calibration_summary)
calibration_summary.to_csv(AUDIT_TABLE_DIR / "calibration_summary.csv", index=False)


In [ ]:
fig = plt.figure(figsize=(7.0, 5.0))
plt.hist(val_df[RESIDUAL_COL].dropna().values, bins=60, alpha=0.65, label="validation residuals")
plt.axvline(thr_val95, linestyle="--", linewidth=2, label=f"val q95 = {thr_val95:.4f}")
plt.axvline(thr_val98, linestyle="--", linewidth=2, label=f"val q98 = {thr_val98:.4f}")
plt.xlabel("Residual (actual - predicted)")
plt.ylabel("Count")
plt.title("Validation-only residual calibration")
plt.legend()
plt.tight_layout()

calib_fig_path = AUDIT_FIG_DIR / "validation_threshold_calibration.png"
plt.savefig(calib_fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", calib_fig_path)


## 3. Leakage-aware ablation

We compare two models trained on the same train / validation / test split from the shipped scored table:

- **clean model**: excludes direct target-construction energy components
- **leaky model**: includes those direct components

In [ ]:
XGB_PARAMS = {
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 1,
    "reg_lambda": 3.0,
    "n_estimators": 391,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

TARGET_TRANSFORM = "log1p"

train_df = scored_df.loc[scored_df["split"] == "train"].copy()
val_df = scored_df.loc[scored_df["split"] == "val"].copy()
test_df_leak = scored_df.loc[scored_df["split"] == "test"].copy()

CLEAN_NUMERIC_FEATURES = [c for c in RAW_NUMERIC_FEATURES if c not in DIRECT_TARGET_COMPONENTS]
LEAKY_NUMERIC_FEATURES = list(RAW_NUMERIC_FEATURES)

def fit_and_score_variant(model_numeric_features: list[str], label: str) -> tuple[dict, pd.DataFrame]:

    X_train, numeric_fill, design_columns = make_design(
        train_df,
        model_numeric_features=model_numeric_features,
        categorical_features=CATEGORICAL_FEATURES,
    )

    X_val, _, _ = make_design(
        val_df,
        model_numeric_features=model_numeric_features,
        categorical_features=CATEGORICAL_FEATURES,
        design_columns=design_columns,
        numeric_fill=numeric_fill,
    )
    
    X_test, _, _ = make_design(
        test_df_leak,
        model_numeric_features=model_numeric_features,
        categorical_features=CATEGORICAL_FEATURES,
        design_columns=design_columns,
        numeric_fill=numeric_fill,
    )

    y_train = train_df[TARGET].to_numpy(dtype=float)
    y_val = val_df[TARGET].to_numpy(dtype=float)
    y_test = test_df_leak[TARGET].to_numpy(dtype=float)

    model = XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, transform_target(y_train, TARGET_TRANSFORM), verbose=False)

    val_pred = inverse_transform_target(model.predict(X_val), TARGET_TRANSFORM)
    test_pred = inverse_transform_target(model.predict(X_test), TARGET_TRANSFORM)

    val_metrics = evaluate_regression(y_val, val_pred)
    test_metrics = evaluate_regression(y_test, test_pred)

    summary = {
        "variant": label,
        "n_model_features": int(X_train.shape[1]),
        "val_mae": val_metrics["mae"],
        "val_rmse": val_metrics["rmse"],
        "val_r2": val_metrics["r2"],
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "test_r2": test_metrics["r2"],
    }

    scored = test_df_leak.loc[:, ["trip_id", "VehId", TARGET]].copy()
    scored[f"pred_{label}"] = test_pred
    scored[f"residual_{label}"] = scored[TARGET] - scored[f"pred_{label}"]
    return summary, scored

clean_summary, clean_scored = fit_and_score_variant(CLEAN_NUMERIC_FEATURES, "clean")
leaky_summary, leaky_scored = fit_and_score_variant(LEAKY_NUMERIC_FEATURES, "leaky")

leakage_comparison = pd.DataFrame([clean_summary, leaky_summary])
display(leakage_comparison)

leakage_comparison.to_csv(AUDIT_TABLE_DIR / "leakage_ablation.csv", index=False)


In [ ]:
plot_df = leakage_comparison.melt(
    id_vars=["variant", "n_model_features"],
    value_vars=["test_mae", "test_rmse", "test_r2"],
    var_name="metric",
    value_name="value",
)

fig = plt.figure(figsize=(8.0, 5.0))
for metric in ["test_mae", "test_rmse", "test_r2"]:
    part = plot_df.loc[plot_df["metric"] == metric]
    plt.plot(part["variant"], part["value"], marker="o", label=metric)

plt.title("Leakage ablation on the test split")
plt.ylabel("Metric value")
plt.legend()
plt.tight_layout()

leak_fig_path = AUDIT_FIG_DIR / "leakage_ablation.png"
plt.savefig(leak_fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", leak_fig_path)


## 4. LIME stability across random seeds

We keep the design-matrix representation for LIME because it is fully compatible with the committed baseline artifact.

The point of the audit is not to find one "perfect" explanation, but to quantify whether the top local features
are stable under perturbation randomness.


In [ ]:
MODEL_NUMERIC_FEATURES = [c for c in RAW_NUMERIC_FEATURES if c not in DIRECT_TARGET_COMPONENTS]

X_all, _, _ = make_design(
    scored_df,
    model_numeric_features=MODEL_NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    design_columns=artifact["feature_columns"],
    numeric_fill=artifact["feature_medians"],
)

feature_columns = artifact["feature_columns"]
background_design = artifact["lime_background"].copy()
test_mask = scored_df["split"].eq("test")
test_scored_df = scored_df.loc[test_mask].reset_index(drop=True)
X_test = X_all.loc[test_mask].reset_index(drop=True)

def black_box_predict_design(df_design: pd.DataFrame) -> np.ndarray:
    aligned = df_design.reindex(columns=feature_columns, fill_value=0.0)
    return np.asarray(artifact["model"].predict(aligned), dtype=float)

stability_cases = (
    test_scored_df.loc[test_scored_df["is_anomaly"]]
    .sort_values(RESIDUAL_COL, ascending=False)
    .drop_duplicates(subset=["VehId"])
    .head(5)
    .reset_index(drop=True)
)

seeds = [0, 1, 2, 3, 4, 5, 6, 7]
stability_rows = []

for _, case_row in stability_cases.iterrows():
    trip_id = str(case_row["trip_id"])
    idx = int(test_scored_df.index[test_scored_df["trip_id"].astype(str) == trip_id][0])
    x0_design = X_test.iloc[idx].copy()

    feature_lists = []
    contribution_frames = []
    local_r2_values = []
    local_rmse_values = []

    for seed in seeds:
        exp = explain_instance(
            trip_id=trip_id,
            x0=x0_design,
            background_df=background_design,
            feature_cols=feature_columns,
            black_box_predict=black_box_predict_design,
            n_samples=3000,
            kernel_width=0.75 * np.sqrt(max(len(feature_columns), 1)),
            ridge_alpha=0.1,
            top_k=5,
            random_state=seed,
        )
        
        local_r2_values.append(exp.local_r2)
        local_rmse_values.append(exp.local_rmse)

        contrib_df = pd.DataFrame(exp.top_features, columns=["feature", "contribution"])
        feature_lists.append(contrib_df["feature"].tolist())
        contribution_frames.append(contrib_df)

    stability_rows.append({
        "trip_id": trip_id,
        "mean_jaccard_top5": mean_pairwise_jaccard(feature_lists),
        "mean_sign_consistency": mean_sign_consistency(contribution_frames),
        "mean_local_r2": float(np.mean(local_r2_values)),
        "mean_local_rmse": float(np.mean(local_rmse_values)),
    })

lime_stability_df = pd.DataFrame(stability_rows).sort_values("mean_jaccard_top5", ascending=False).reset_index(drop=True)
display(lime_stability_df)

lime_stability_df.to_csv(AUDIT_TABLE_DIR / "lime_stability_summary.csv", index=False)


In [ ]:
fig = plt.figure(figsize=(7.5, 5.0))
x = np.arange(len(lime_stability_df))
width = 0.38

plt.bar(x - width / 2, lime_stability_df["mean_jaccard_top5"], width=width, label="mean Jaccard(top-5)")
plt.bar(x + width / 2, lime_stability_df["mean_sign_consistency"], width=width, label="mean sign consistency")
plt.xticks(x, lime_stability_df["trip_id"], rotation=45, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Stability score")
plt.title("LIME stability across random seeds")
plt.legend()
plt.tight_layout()

stability_fig_path = AUDIT_FIG_DIR / "lime_stability.png"
plt.savefig(stability_fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", stability_fig_path)


## 5. Subgroup robustness audit

For this project, the most honest fairness-style statement is **operational subgroup robustness**:
do errors and anomaly rates stay similar across technical subgroups such as transmission or vehicle type?


In [ ]:
robustness_df = test_df.copy()
robustness_df["is_anomaly_calibrated"] = test_df["is_anomaly_val95"]

group_tables = {}

for group_col in ["Transmission", "VehicleType", "Drive Wheels"]:
    if group_col in robustness_df.columns:
        group_tables[group_col] = subgroup_metrics(robustness_df, group_col, "is_anomaly_calibrated")
        group_tables[group_col].to_csv(AUDIT_TABLE_DIR / f"subgroup_{group_col.replace(' ', '_')}.csv", index=False)
        print(f"\nGroup audit for {group_col}")
        display(group_tables[group_col].head(10))


In [ ]:
group_col = "Transmission"

if group_col in group_tables:
    plot_df = group_tables[group_col].copy().head(8)
    plt.figure(figsize=(8.0, 5.0))
    plt.bar(plot_df["group_value"].astype(str), plot_df["anomaly_rate"])
    plt.xticks(rotation=35, ha="right")
    plt.ylabel("Calibrated anomaly rate")
    plt.title("Subgroup robustness by Transmission")
    plt.tight_layout()

    subgroup_fig_path = AUDIT_FIG_DIR / "subgroup_transmission_anomaly_rate.png"
    plt.savefig(subgroup_fig_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", subgroup_fig_path)
